# 02 · Feature engineering: feature table v1

Owner: Supriye (S10–S20). Loads `data/processed/features_v1.parquet` (built by `python -m src.features v1`), shows what is in it, draws figure **F3** (S17 step 5) and runs the **five validation and leakage checks of S19**. The results go to `results/tables/v1_validation_checks.csv`.

Definitions of every column: `docs/feature_dictionary.md`. Run with Restart & Run All.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import baseline
from src.config import FIGURES_DIR, PROCESSED_DIR, TABLES_DIR
from src.data_loader import load_device, load_email, load_file, load_insiders, load_logon
from src.features import META_COLUMNS, behavior_checks, feature_columns, get_model_matrix, is_after_hours

df = pd.read_parquet(PROCESSED_DIR / 'features_v1.parquet')
features = feature_columns(df)
GROUP = {'logon': 'logon', 'logoff': 'logon', 'first': 'logon', 'last': 'logon', 'n': 'logon',
         'work': 'logon', 'afterhours': 'time', 'weekend': 'time', 'ctx': 'context'}
def group(c):
    if c.startswith(('z_user_', 'z_peer_')):
        return c[:6]
    if c.startswith('new_'):
        return 'new'
    token = c.split('_')[1]
    return GROUP.get(token, token)
groups = pd.Series([group(c) for c in features])
print(f'{len(df):,} user-days, {df.user.nunique():,} users, {len(features)} model inputs, '
      f'{int(df.label.sum())} malicious ({df.label.mean():.3%})')
groups.value_counts().rename('columns').to_frame()

330,268 user-days, 1,000 users, 76 model inputs, 984 malicious (0.298%)


,columns
z_user,15
z_peer,15
email,8
http,8
logon,7
file,6
context,5
time,4
usb,4
new,4


## Behavior checks (S12, S14, S15 "Done when")

For each scenario: a feature's mean on the malicious days of the labeled accounts vs. the same accounts' other days.

In [2]:
behavior_checks(df).round(3)

,scenario,feature,accounts,mean_malicious_days,mean_other_days,share_malicious_days_gt0
0,1,f_usb_connect,30,1.289,0.043,0.807
1,1,f_http_leak,30,0.807,0.000,0.795
2,1,f_afterhours_logon,30,0.964,0.112,0.783
3,2,f_http_job,30,2.492,0.855,0.739
4,2,f_http_competitor,30,1.772,0.318,0.656
5,2,f_usb_connect,30,3.120,2.065,0.945
6,3,f_email_recipients,12,28.650,25.299,1.000
7,3,f_http_hack,12,1.175,0.000,0.250
8,3,f_logon_other_pc,12,1.950,2.257,0.500


## Figure F3: one insider against their own baseline (S17 step 5)

In [3]:
# Illustration only: the real insider (>= 5 malicious days, >= 60 active days before the first one)
# whose malicious days stand out most against their own USB baseline.
mal = df[(df.label == 1) & (df.insider == 1)]
first_bad = mal.groupby('user').date.min()
history = df.merge(first_bad.rename('first_bad'), on='user')
history = history[history.date < history.first_bad].groupby('user').size()
stats = mal.groupby('user').agg(days=('label', 'size'), scenario=('scenario', 'first'),
                                mean_z=('z_user_usb_connect', 'mean'))
stats = stats[(stats.days >= 5) & (history.reindex(stats.index).fillna(0) >= 60)]
user = stats.mean_z.idxmax()
print(user, stats.loc[user].to_dict())
fig = baseline.plot_user_baseline(df, user, 'f_usb_connect')
fig

RMW0542 {'days': 29.0, 'scenario': 2.0, 'mean_z': 0.47611949899475864}


<Figure size 800x320 with 1 Axes>

## S19 validation and leakage checks

**Check 1:** no missing or infinite values in any model-input column.

In [4]:
values = df[features].to_numpy(dtype=float)
n_nan, n_inf = int(np.isnan(values).sum()), int(np.isinf(values).sum())
check1 = n_nan == 0 and n_inf == 0
print('missing:', n_nan, ' infinite:', n_inf, ' ->', 'PASS' if check1 else 'FAIL')

missing: 0  infinite: 0  -> PASS


**Check 2:** rows per split, malicious rows per split, and malicious rows per split × scenario (every cell > 0).

In [5]:
per_split = df.groupby('split').agg(rows=('label', 'size'), malicious=('label', 'sum'))
cells = (df[df.label == 1].pivot_table(index='split', columns='scenario', values='user', aggfunc='count')
         .reindex(index=['train', 'val', 'test'], columns=[1, 2, 3]).fillna(0).astype(int))
check2 = bool((cells > 0).all().all())
print('->', 'PASS' if check2 else 'FAIL')
per_split.join(cells.add_prefix('malicious_s'))

-> PASS


,rows,malicious,malicious_s1,malicious_s2,malicious_s3
split,,,,,
test,65007,170,9,157,4
train,209072,578,58,492,28
val,56189,236,16,212,8


**Check 3 (leakage):** recompute the per-user baselines after deleting the last month. The `z_user` values of every earlier row must not change.

In [6]:
raw = df[META_COLUMNS + [c for c in df.columns if c.startswith('f_')]]
max_change = baseline.leakage_check(raw)
check3 = max_change == 0.0
print('largest change in an earlier z_user value:', max_change, '->', 'PASS' if check3 else 'FAIL')

largest change in an earlier z_user value: 0.0 -> PASS


**Check 4 (hand check, human-only):** for 3 insiders, one per scenario, open the raw logs for one malicious day and count 3–4 feature values by hand. Supriye records the result here.

*Observation (Supriye):* 

To support the hand check, the cell below recounts the same values directly from the raw logs with plain pandas (independent of `src/features.py`).

In [7]:
ins = load_insiders()
picks = []
for scenario in (1, 2, 3):
    users = set(ins.loc[ins.scenario == scenario, 'user'])
    row = df[(df.scenario == scenario) & df.user.isin(users)].sort_values(['user', 'date']).iloc[0]
    picks.append(row)
rows = []
for row in picks:
    u, day = row.user, row.date
    lg = load_logon(columns=['user', 'date', 'activity'], users=[u]); lg = lg[lg.day == day]
    dv = load_device(columns=['user', 'date', 'activity'], users=[u]); dv = dv[dv.day == day]
    fl = load_file(columns=['user', 'date', 'filename'], users=[u]); fl = fl[fl.day == day]
    em = load_email(columns=['user', 'date', 'to'], users=[u]); em = em[em.day == day]
    recount = {'f_logon_count': (lg.activity == 'Logon').sum(),
               'f_afterhours_logon': ((lg.activity == 'Logon') & is_after_hours(lg.hour)).sum(),
               'f_usb_connect': (dv.activity == 'Connect').sum(), 'f_file_copy': len(fl),
               'f_email_sent': len(em)}
    for col, n in recount.items():
        rows.append({'scenario': row.scenario, 'user': u, 'date': day.date(), 'feature': col,
                     'table': int(row[col]), 'raw_log': int(n)})
recount = pd.DataFrame(rows)
recount_ok = bool((recount.table == recount.raw_log).all())
print('automated recount ->', 'MATCH' if recount_ok else 'MISMATCH')
recount

automated recount -> MATCH


,scenario,user,date,feature,table,raw_log
0,1,AAM0658,2010-10-23,f_logon_count,1,1
1,1,AAM0658,2010-10-23,f_afterhours_logon,1,1
2,1,AAM0658,2010-10-23,f_usb_connect,1,1
3,1,AAM0658,2010-10-23,f_file_copy,0,0
4,1,AAM0658,2010-10-23,f_email_sent,0,0
5,2,AAF0535,2010-06-28,f_logon_count,1,1
6,2,AAF0535,2010-06-28,f_afterhours_logon,0,0
7,2,AAF0535,2010-06-28,f_usb_connect,3,3
8,2,AAF0535,2010-06-28,f_file_copy,2,2
9,2,AAF0535,2010-06-28,f_email_sent,2,2


**Check 5:** no column derived from `label`, `scenario` or future LDAP months.

- Model inputs are only `f_*`, `z_user_*`, `z_peer_*`, `new_*` (`get_model_matrix()` refuses any other column).
- No input is named after or computed from the labels: the feature code never reads them, and the inputs below are checked by name.
- `role` and `department` come from the LDAP snapshot of the row's own month (`merge_asof(direction='backward')`), never a later one. They are metadata, not inputs.
- No "days until leaving" feature (S16 step 3).

In [8]:
forbidden = ('label', 'scenario', 'insider', 'leav', 'role', 'department', 'split')
bad_names = [c for c in features if any(w in c for w in forbidden)]
try:
    get_model_matrix(df, columns=features + ['label']); guard = False
except ValueError:
    guard = True
check5 = not bad_names and guard
print('suspicious input names:', bad_names, '| guard rejects label:', guard, '->', 'PASS' if check5 else 'FAIL')

suspicious input names: [] | guard rejects label: True -> PASS


## Summary

In [9]:
summary = pd.DataFrame([
    {'check': '1 no missing / infinite inputs', 'result': 'PASS' if check1 else 'FAIL'},
    {'check': '2 every split x scenario has malicious rows', 'result': 'PASS' if check2 else 'FAIL'},
    {'check': '3 baselines unchanged after deleting the last month', 'result': 'PASS' if check3 else 'FAIL'},
    {'check': '4 hand check (human-only); automated recount', 'result': 'recount ' + ('MATCH' if recount_ok else 'MISMATCH')},
    {'check': '5 no label-derived or future columns', 'result': 'PASS' if check5 else 'FAIL'},
])
summary.to_csv(TABLES_DIR / 'v1_validation_checks.csv', index=False)
summary

,check,result
0,1 no missing / infinite inputs,PASS
1,2 every split x scenario has malicious rows,PASS
2,3 baselines unchanged after deleting the last ...,PASS
3,4 hand check (human-only); automated recount,recount MATCH
4,5 no label-derived or future columns,PASS
